# Words per character

How many words does each character say in the Grimm fairy tales?

The tales are encoded in TEI XML (`data/fairy-tales/`):
- in the **header**, `<listPerson>` lists the characters of the tale, each with an `xml:id` and a `gender`:

  `<person xml:id="redcap-wolf" gender="male"><persName>wolf</persName></person>`
- in the **text**, every speech is in a `<said>` element, and `who` points to the speaker:

  `<said who="#redcap-wolf">‘Good day, Little Red-Cap,’</said>`

We count the words in the `<said>` of each character and save the result in a CSV file: one row per character, with the number of words and the gender.

## 1. Find the files

In [14]:
from pathlib import Path  # Tools for files and folders.
import re  # Regular expressions, to find the words.
import xml.etree.ElementTree as ET  # Reads XML files (part of Python, nothing to install).
import pandas as pd  # Tables (DataFrames) and CSV files.

# The tales are the files whose names start with a number (e.g. "07_briar-rose.xml").
# This skips the file with the whole collection, which would count every tale twice.
files = sorted(Path("../data/fairy-tales").glob("[0-9]*.xml"))
print(len(files), "tales")

ns = {"tei": "http://www.tei-c.org/ns/1.0"}  # TEI elements live in this namespace: we must give it to every search.

22 tales


## 2. Count the words of a text

We look for sequences of letters. An apostrophe or a hyphen between letters stays inside the word (*king’s*, *Red-Cap* = 1 word each), while the double hyphen used as a dash separates words (*me--you* = 2 words).

In [2]:
def count_words(text):
    """Return the number of words in a text."""
    words = re.findall(r"[A-Za-z]+(?:[’'-][A-Za-z]+)*", text)  # Letters, optionally joined by ’ ' or - to more letters.
    return len(words)

print(count_words("‘I know what your wish is, and it shall be fulfilled--you will soon have a daughter.’"))
print(count_words("The king’s daughter, Little Red-Cap"))

17
5


## 3. Try with one tale

Before processing all the tales, let's look at one: first the characters in the header, then the speeches.

In [3]:
root = ET.parse(files[0]).getroot()  # Read the first file; "root" is the <TEI> element.

title = root.find(".//tei:titleStmt/tei:title", ns).text  # The title in the header.
print("Title:", title)
print()

# The characters: <person> (one character) or <personGrp> (a group, e.g. the seven dwarfs).
characters = root.findall(".//tei:listPerson/*", ns)  # "*" = every element inside <listPerson>.
for character in characters:
    char_id = character.get("{http://www.w3.org/XML/1998/namespace}id")  # xml:id has its own namespace.
    gender = character.get("gender")
    name = character.find("tei:persName", ns).text  # The first name in the list.
    print(char_id, "|", name, "|", gender)

Title: Briar Rose

briarrose-fish | fish | unspecified
briarrose-queen | queen | female
briarrose-thirteenth_fairy | thirteenth fairy | female
briarrose-briar_rose | Briar Rose | female
briarrose-old_lady | old lady | female
briarrose-prince | prince | male


In [4]:
# The first 5 speeches of the tale.
speeches = root.findall(".//tei:said", ns)
print(len(speeches), "speeches")
print()

for said in speeches[:5]:
    text = "".join(said.itertext())  # All the text inside <said>, also inside <persName>.
    print(said.get("who"), "->", count_words(text), "words:", " ".join(text.split()))  # " ".join(...split()) removes line breaks.

8 speeches

#briarrose-fish -> 24 words: ‘I know what your wish is, and it shall be fulfilled, in return for your kindness to me--you will soon have a daughter.’
#briarrose-queen -> 17 words: ‘I will have the fairies also, that they might be kind and good to our little daughter.’
#briarrose-thirteenth_fairy -> 17 words: ‘The king’s daughter shall, in her fifteenth year, be wounded by a spindle, and fall down dead.’
#briarrose-briar_rose -> 5 words: ‘Why, how now, good mother,’
#briarrose-briar_rose -> 5 words: ‘what are you doing there?’


## 4. All the tales

Now we do the same for every tale. For each tale:
1. we take the characters from the header and start each one at 0 words, so that characters who never speak are in the table too;
2. we go through the speeches and add their words to the speaker.

One speech has two speakers (`who="#hanselgretel-hansel #hanselgretel-gretel"`, Hansel and Gretel speaking together): the words are counted for both.

In [5]:
rows = []  # One row per character, for all the tales.
speech_rows = []  # One row per speech, for all the tales (used in section 7).

for f in files:
    root = ET.parse(f).getroot()
    title = root.find(".//tei:titleStmt/tei:title", ns).text

    # 1. The characters of this tale, each starting at 0 words.
    words = {}  # character id -> number of words
    names = {}  # character id -> name
    genders = {}  # character id -> gender
    for character in root.findall(".//tei:listPerson/*", ns):
        char_id = character.get("{http://www.w3.org/XML/1998/namespace}id")
        words[char_id] = 0
        names[char_id] = character.find("tei:persName", ns).text
        genders[char_id] = character.get("gender")

    # 2. The speeches: add the words to the speaker(s).
    for said in root.findall(".//tei:said", ns):
        text = " ".join("".join(said.itertext()).split())  # The text of the speech, without line breaks.
        n = count_words(text)
        speech_rows.append({"speech_id": len(speech_rows), "tale": title, "speaker": said.get("who"), "speech": text, "words": n})  # Keep every speech, for the ground truth.
        for speaker in said.get("who").split():  # Usually one speaker; split() handles "#a #b".
            char_id = speaker.lstrip("#")  # "#redcap-wolf" -> "redcap-wolf"
            words[char_id] = words[char_id] + n

    # 3. One row per character.
    for char_id in words:
        rows.append({
            "tale": title,
            "character": char_id,
            "name": names[char_id],
            "gender": genders[char_id],
            "words": words[char_id],
        })

df = pd.DataFrame(rows)  # Turn the list of rows into a table.
print(len(df), "characters")
df.head(10)

139 characters


,tale,character,name,gender,words
0,Briar Rose,briarrose-fish,fish,unspecified,24
1,Briar Rose,briarrose-queen,queen,female,17
2,Briar Rose,briarrose-thirteenth_fairy,thirteenth fairy,female,17
3,Briar Rose,briarrose-briar_rose,Briar Rose,female,17
4,Briar Rose,briarrose-old_lady,old lady,female,1
5,Briar Rose,briarrose-prince,prince,male,14
6,The Frog-Prince,frogprince-princess,princess,female,170
7,The Frog-Prince,frogprince-frog,frog,male,227
8,The Frog-Prince,frogprince-king,king,male,16
9,The Frog-Prince,frogprince-heinrich,Heinrich,male,0


## 5. Check and save

In [6]:
# The 10 characters who speak the most.
df.sort_values("words", ascending=False).head(10)

,tale,character,name,gender,words
138,Snow-White and Rose-Red,snowwhiterosered-dwarf,dwarf,male,328
124,King Grisly-Beard,grislybeard-king_grisly-beard,Grisly-beard,male,326
53,Clever Gretel,clevergretel-gretel,Gretel,female,322
26,Hansel and Gretel,hanselgretel-stepmother,stepmother,female,295
43,The Robber Bridegroom,robber-millers_daughter,millers daughter,female,288
76,Clever Elsie,cleverelsie-clever_elsie,Clever Elsie,female,245
96,Lily and the Lion,lilylion-lily,Lily,female,229
7,The Frog-Prince,frogprince-frog,frog,male,227
23,Hansel and Gretel,hanselgretel-hansel,Hansel,male,212
137,Snow-White and Rose-Red,snowwhiterosered-mr_bear,Mr Bear,male,190


In [7]:
# A quick check: total words and number of characters per gender.
df.groupby("gender")["words"].agg(["sum", "count"])

,sum,count
gender,,
female,4929,52
male,3939,64
unspecified,953,23


In [8]:
df.to_csv("words_per_character.csv", index=False)  # index=False: do not save the row numbers.
print("Saved", len(df), "rows in words_per_character.csv")

Saved 139 rows in words_per_character.csv


## 6. The five characters who talk the most in each tale

For each tale, we keep only its rows, sort them by number of words, and print the first five. Characters who never speak (0 words) are left out.

In [9]:
for title in df["tale"].unique():  # Every tale, once, in the order of the files.
    tale = df[df["tale"] == title]  # Only the characters of this tale.
    tale = tale[tale["words"] > 0]  # Leave out the characters who never speak.
    top5 = tale.sort_values("words", ascending=False).head(5)  # The 5 with the most words.

    print(title)
    for i, row in top5.iterrows():
        print("   ", row["name"], "(" + row["gender"] + "):", row["words"], "words")
    print()  # Empty line between tales.

Briar Rose
    fish (unspecified): 24 words
    queen (female): 17 words
    thirteenth fairy (female): 17 words
    Briar Rose (female): 17 words
    prince (male): 14 words

The Frog-Prince
    frog (male): 227 words
    princess (female): 170 words
    king (male): 16 words

The Goose-Girl
    princess (female): 170 words
    waiting-maid (female): 151 words
    Curdken (male): 85 words
    Falada (unspecified): 36 words
    old king (male): 33 words

Rapunzel
    Dame Gothel (female): 158 words
    Rapunzel (female): 92 words
    husband (male): 72 words
    kings son (male): 31 words
    wife (female): 20 words

Hansel and Gretel
    stepmother (female): 295 words
    Hansel (male): 212 words
    witch (female): 176 words
    Gretel (female): 150 words
    father (male): 129 words

Mother Holle
    Mother Holle (female): 118 words
    loaves (unspecified): 42 words
    lazy girl (female): 35 words
    apple tree (unspecified): 26 words
    Cinderella (female): 26 words

Little Red

## 7. Ground truth

The results depend on checking by hand the **word count**.

The code below chooses the samples at random and saves them as CSV files with an empty column, to fill in by hand (e.g. in a spreadsheet, saving again as CSV):
- `ground_truth_words.csv`: 20 speeches. Count the words by hand and write the number in `words_by_hand`.

Write your doubts in `notes`.

**Attention**: a file is created only if it does not exist yet, so your annotations are never overwritten.

`random_state=1` makes the choice random but repeatable: every time we run the code, we get the same sample.

In [15]:

speeches_df = pd.DataFrame(speech_rows)  # All the speeches of all the tales.
words_file = Path("ground_truth_words.csv")
if not words_file.exists():
    sample = speeches_df.sample(n=20, random_state=1)  # 20 speeches at random.
    sample = sample[["speech_id", "tale", "speech"]].copy()  # Without the count of the code.
    sample["words_by_hand"] = ""
    sample["notes"] = ""
    sample.to_csv(words_file, index=False)
    print("Created", words_file, "- now annotate it!")
else:
    print(words_file, "already exists: not overwritten.")

ground_truth_words.csv already exists: not overwritten.


## 8. Validation

When the ground-truth file is annotated, run the cells below. If a file is not annotated yet, the cell says so.



### Words: difference between code and hand

In [16]:
gt = pd.read_csv("ground_truth_words.csv")

if gt["words_by_hand"].isna().any():
    print("Annotate all the rows of ground_truth_words.csv first.")
else:
    merged = gt.merge(speeches_df[["speech_id", "words"]], on="speech_id")  # Add the count of the code.
    merged["difference"] = merged["words"] - merged["words_by_hand"]  # Positive: the code counts more.
    exact = merged[merged["difference"] == 0]
    print("Exact counts:", len(exact), "of", len(merged))
    print()
    print("Differences (read the speech to explain them):")
    print(merged[merged["difference"] != 0][["speech", "words", "words_by_hand", "difference"]])

Annotate all the rows of ground_truth_words.csv first.
